# Frozen encoder, learnable embeddings — does the encoder need to learn anything?

`emb_rgcn` (learnable node table + trained R-GCN) and `emb_mlp` (learnable table + trained MLP) both
train **two** things at once: the per-node embeddings and the encoder weights. This grid removes the
second. The encoder's weights are initialised and then **frozen at their random values for the whole
run** — pre-training and fine-tuning. Only the node embedding table (and the DistMult decoder) learns.

If a *random, untrained* R-GCN with learnable embeddings matches the fully trained one, then the
relation-specific weight matrices are not learning anything the model needs: they are acting as a
fixed random mixing of neighbours, and every bit of fitted knowledge lives in the node table.

| arm | encoder | $W_r$ trained? | table trained? | trainable params | status |
|---|---|---|---|---|---|
| `rgcn` | R-GCN | yes | — (frozen features) | 1,015,000 | imported |
| `emb_rgcn` | R-GCN | yes | yes | 13,946,200 | imported |
| `emb_mlp` | MLP | yes | yes | 12,956,400 | imported |
| **`emb_rgcn_frozenW`** | R-GCN | **no** | yes | 12,936,200 | **new** |
| **`emb_mlp_frozenW`** | MLP | **no** | yes | 12,936,200 | **new** |
| **`emb_rgcn_frozenW_matched`** | R-GCN | **no** | yes, wider | 13,970,696 | **new** |

### What exactly is frozen

`HeteroRGCNLayer` holds one `nn.Linear(n_inp, n_hid)` and one `nn.Linear(n_hid, n_out)` **per edge
type** — these are the paper's $W_{r,M}^{(l)}$, 50 of each, 1,010,000 parameters. Those are what get
`requires_grad = False`. The MLP arms freeze their single shared `Linear` per layer the same way.

The DistMult decoder's $\mathbf{w}_r$ (`w_rels`, a 50×100 *vector* per relation, 5,000 parameters)
stays **trainable**. It is the decoder, not the encoder, and with it frozen too nothing could learn
which relation is being scored. `FREEZE_DECODER_W` flips that if you want the stricter version.

The frozen weights are verified unchanged after training, not assumed: §6 records
`encoder_weight_delta`, the max absolute change in any encoder parameter between initialisation and
the end of the run. It must be exactly 0.

### Capacity matching

Freezing $W_r$ does not change the parameter *count*, only how many parameters are *trainable*. For
the R-GCN arms that is a real cut — 1,010,000 of 13,946,200, about 7% — so the matched arm widens the
embedding table to compensate. The only free dimension is `n_inp`, since `n_hid = n_out` is fixed by
the decoder:

$$129{,}312 \times n_{\text{inp}} = 129{,}312 \times 100 + 1{,}010{,}000 \;\Rightarrow\; n_{\text{inp}} = 107.81$$

so `n_inp = 108` lands at 13,965,696 trainable table parameters, +0.18% of target. Widening `n_inp`
grows the *frozen* encoder too (to 1,050,000), which costs nothing in trainable terms.

For the MLP arms the deficit is only 20,200 of 12,956,400 — **0.16%** — and `n_inp = 101` would
overshoot by 0.84%. `n_inp = 100` is already the closest integer, so there is no matched MLP arm; the
unmatched one is the matched one.

### What this can show

- **`emb_rgcn_frozenW` vs `emb_rgcn`** — the value of *training* the encoder, with the table learnable
  in both. This is the headline contrast.
- **`emb_rgcn_frozenW` vs `emb_mlp_frozenW`** — message passing through a *random* projection. Both
  arms have an identical trainable budget (12,936,200), so this is capacity-matched by construction.
- **`emb_mlp_frozenW` vs `emb_mlp`** — the same question for a model with no message passing at all.
  Here the frozen encoder is a single random `Linear`, so a null result says the 20,200 weights were
  doing nothing.

---
## 1 · Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))   # PyG port FIRST, so `import txgnn` never hits dgl_implementation/

import torch
import torch.nn as nn

import txgnn
from txgnn import TxData, TxGNN
from txgnn.model import HeteroRGCN, HeteroRGCNLayer
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit MODEL_CFG.')

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo  :', REPO_ROOT)
print('txgnn :', txgnn.__file__)
print('torch :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 · Configuration

Seeds, split and schedule match `ablation_learnable_embeddings`, so `rgcn`, `emb_rgcn` and `emb_mlp` import cleanly and every delta is paired per seed.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES_DEFAULT = ['disease_disease', 'rev_disease_protein']

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_frozen_encoder'
EMB_ROOT    = REPO_ROOT / 'results' / 'ablation_learnable_embeddings'   # read-only
SWAP_ROOT   = REPO_ROOT / 'results' / 'ablation_encoder_swap'           # read-only
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM = 'emb_rgcn'      # the arm every frozen-W delta is measured against

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True
IMPORT_REFS       = True
PROBE_EMB_LEARNS  = True    # confirm the table receives gradient (costs 1 finetune epoch)
FREEZE_DECODER_W  = False   # also freeze the DistMult w_rels - see section 0
STOP_AFTER_N_RUNS = None

print('split  :', SPLIT)
print('seeds  : %d -> %s' % (len(SEEDS), SEEDS))
print('results:', ABL_ROOT.relative_to(REPO_ROOT))
print('decoder w_rels frozen:', FREEZE_DECODER_W)

---
## 3 · The encoders

Carried over from `ablation_learnable_embeddings` unchanged, so the imported arms mean what they did
there. The only addition is `freeze_encoder`, applied after `model_initialize` and before any
training.

In [ ]:
class MLPLayer(nn.Module):
    '''No message passing: h[ntype] = W x[ntype]. Verbatim from ablation_encoder_swap.ipynb.'''

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = nn.Linear(in_size, out_size)
        self.in_size, self.out_size = in_size, out_size

    def forward(self, G, feat_dict):
        return {nt: self.weight(feat_dict[nt]) for nt in G.node_types}


_TXGNN_MODULE    = sys.modules['txgnn.TxGNN']
_ORIG_HETERORGCN = HeteroRGCN
LAYER_CLASSES    = {'rgcn': HeteroRGCNLayer, 'mlp': MLPLayer}


def make_encoder_class(layer_kind, learnable_emb, kind):
    layer_cls = LAYER_CLASSES[layer_kind]

    class SwappedHeteroRGCN(_ORIG_HETERORGCN):
        '''HeteroRGCN with an optional layer swap and an optional learnable node-embedding table.'''

        def __init__(self, G, **kw):
            super().__init__(G, **kw)
            if layer_kind != 'rgcn':
                self.layer1 = layer_cls(kw['in_size'], kw['hidden_size'])
                self.layer2 = layer_cls(kw['hidden_size'], kw['out_size'])
            self.encoder_kind  = kind
            self.learnable_emb = learnable_emb
            self.node_emb, self._emb_key = None, {}
            if learnable_emb:
                self.node_emb = nn.ParameterDict()
                for nt in G.node_types:
                    k = nt.replace('/', '__')          # ParameterDict keys cannot contain '.'
                    self._emb_key[nt] = k
                    p = nn.Parameter(torch.empty(G[nt].num_nodes, kw['in_size']))
                    nn.init.xavier_uniform_(p)         # same init as initialize_node_embedding
                    self.node_emb[k] = p

        def _bind_embeddings(self, G):
            '''Point G[ntype].inp at the live Parameters. Must run inside every forward.'''
            if not self.learnable_emb:
                return
            for nt in G.node_types:
                G[nt].inp = self.node_emb[self._emb_key[nt]]

        def forward(self, G, *a, **kw):
            self._bind_embeddings(G)
            return super().forward(G, *a, **kw)

        def forward_minibatch(self, pos_G, neg_G, G, *a, **kw):
            self._bind_embeddings(G)
            return super().forward_minibatch(pos_G, neg_G, G, *a, **kw)

    SwappedHeteroRGCN.__name__ = 'HeteroRGCN_' + kind
    return SwappedHeteroRGCN


ENCODER_SPECS = {
    'rgcn':                     dict(layer='rgcn', emb=False, freeze=False, n_inp=100),
    'emb_rgcn':                 dict(layer='rgcn', emb=True,  freeze=False, n_inp=100),
    'emb_mlp':                  dict(layer='mlp',  emb=True,  freeze=False, n_inp=100),
    'emb_rgcn_frozenW':         dict(layer='rgcn', emb=True,  freeze=True,  n_inp=100),
    'emb_mlp_frozenW':          dict(layer='mlp',  emb=True,  freeze=True,  n_inp=100),
    'emb_rgcn_frozenW_matched': dict(layer='rgcn', emb=True,  freeze=True,  n_inp=None),  # solved below
}


def install_encoder(kind):
    spec = ENCODER_SPECS[kind]
    cls = (_ORIG_HETERORGCN if kind == 'rgcn'
           else make_encoder_class(spec['layer'], spec['emb'], kind))
    _TXGNN_MODULE.HeteroRGCN = cls
    return cls


install_encoder('rgcn')
print('encoders:', sorted(ENCODER_SPECS))

In [ ]:
def encoder_parameters(model):
    '''The per-relation message weights - layer1 and layer2. These are the W_r.'''
    return list(model.layer1.parameters()) + list(model.layer2.parameters())


def encoder_params(model):
    return sum(p.numel() for p in encoder_parameters(model))


def node_emb_params(model):
    return sum(p.numel() for p in model.node_emb.parameters()) if getattr(model, 'node_emb', None) else 0


def trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def freeze_encoder(model, freeze_decoder_w=False):
    '''Freeze the encoder's per-relation weights. Returns (n_frozen, snapshot) for verification.'''
    n = 0
    for p in encoder_parameters(model):
        p.requires_grad = False
        n += p.numel()
    if freeze_decoder_w:
        model.w_rels.requires_grad = False
        n += model.w_rels.numel()
    snapshot = [p.detach().clone() for p in encoder_parameters(model)]
    return n, snapshot


def encoder_weight_delta(model, snapshot):
    '''Max |change| in any encoder parameter since the snapshot. Must be 0 for a frozen encoder.'''
    cur = encoder_parameters(model)
    assert len(cur) == len(snapshot), 'encoder structure changed during the run'
    return max(float((a.detach() - b).abs().max()) for a, b in zip(cur, snapshot))

---
## 4 · Arms, and the matched embedding width

`n_inp` for the matched arm is solved from the graph at run time, not hardcoded, and then asserted
against the table the model actually builds.

In [ ]:
# trainable-parameter budget of the reference arm, from its own metadata where available
REF_TRAINABLE = None
_ref_meta = EMB_ROOT / ('%s_seed%d' % (REF_ARM, SEEDS[0])) / 'run_meta.json'
if _ref_meta.exists():
    _m = json.loads(_ref_meta.read_text())
    REF_TRAINABLE = int(_m['n_params'])
    N_NODES = int(_m['n_nodes_total'])
    print('%s: %s trainable params over %s nodes (from %s)'
          % (REF_ARM, format(REF_TRAINABLE, ','), format(N_NODES, ','), _ref_meta.parent.name))
else:
    N_NODES = 129312
    REF_TRAINABLE = N_NODES * MODEL_CFG['n_inp'] + 50 * 2 * (100 * 100 + 100) + 50 * 100
    print('WARNING: %s not found; falling back to the computed budget %s'
          % (_ref_meta, format(REF_TRAINABLE, ',')))


def matched_n_inp(n_nodes, target_trainable, decoder=5000):
    '''Smallest-error n_inp whose embedding table alone carries the reference trainable budget.'''
    want = target_trainable - decoder
    best = None
    for k in range(MODEL_CFG['n_inp'], 4 * MODEL_CFG['n_inp']):
        err = abs(n_nodes * k - want)
        if best is None or err < best[1]:
            best = (k, err)
    k = best[0]
    return k, n_nodes * k + decoder


MATCHED_N_INP, MATCHED_TRAINABLE = matched_n_inp(N_NODES, REF_TRAINABLE)
ENCODER_SPECS['emb_rgcn_frozenW_matched']['n_inp'] = MATCHED_N_INP

print('matched arm: n_inp = %d -> %s trainable (%+.3f%% of %s)'
      % (MATCHED_N_INP, format(MATCHED_TRAINABLE, ','),
         100.0 * (MATCHED_TRAINABLE - REF_TRAINABLE) / REF_TRAINABLE, format(REF_TRAINABLE, ',')))
print('MLP arms: deficit is only %s of %s (%.2f%%), so n_inp stays at %d - see section 0'
      % (format(2 * (100 * 100 + 100), ','), format(N_NODES * 100 + 2 * (100 * 100 + 100) + 5000, ','),
         100.0 * 2 * (100 * 100 + 100) / (N_NODES * 100 + 2 * (100 * 100 + 100) + 5000),
         MODEL_CFG['n_inp']))

In [ ]:
ARMS = [
    {'arm': 'rgcn',     'encoder': 'rgcn',     'source': SWAP_ROOT,
     'note': 'reference: published model, frozen features (imported)'},
    {'arm': 'emb_rgcn', 'encoder': 'emb_rgcn', 'source': EMB_ROOT,
     'note': 'reference: learnable table + TRAINED R-GCN (imported)'},
    {'arm': 'emb_mlp',  'encoder': 'emb_mlp',  'source': EMB_ROOT,
     'note': 'reference: learnable table + TRAINED MLP (imported)'},

    {'arm': 'emb_rgcn_frozenW', 'encoder': 'emb_rgcn_frozenW', 'source': None,
     'note': 'NEW: learnable table, W_r frozen at random init'},
    {'arm': 'emb_mlp_frozenW',  'encoder': 'emb_mlp_frozenW',  'source': None,
     'note': 'NEW: learnable table, shared MLP weight frozen at random init'},
    {'arm': 'emb_rgcn_frozenW_matched', 'encoder': 'emb_rgcn_frozenW_matched', 'source': None,
     'note': 'NEW: as above, table widened so the trainable budget matches emb_rgcn'},
]

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
assert REF_ARM in ARMS_BY_NAME, 'reference arm %r missing' % REF_ARM

IMPORTABLE = {a['arm'] for a in ARMS if a['source'] is not None}
NEW_ARMS   = [a['arm'] for a in ARMS if a['source'] is None]


def cfg_for(kind):
    return dict(MODEL_CFG, n_inp=ENCODER_SPECS[kind]['n_inp'])


rows = []
for a in ARMS:
    sp = ENCODER_SPECS[a['encoder']]
    enc = (50 if sp['layer'] == 'rgcn' else 1) * ((sp['n_inp'] * 100 + 100) + (100 * 100 + 100))
    tbl = N_NODES * sp['n_inp'] if sp['emb'] else 0
    rows.append({'arm': a['arm'], 'layer': sp['layer'], 'message passing': sp['layer'] == 'rgcn',
                 'W_r trained': not sp['freeze'], 'table trained': sp['emb'], 'n_inp': sp['n_inp'],
                 'encoder params': enc, 'table params': tbl,
                 'TRAINABLE': (0 if sp['freeze'] else enc) + tbl + 5000,
                 'source': 'new' if a['source'] is None else Path(a['source']).name})
display(pd.DataFrame(rows).style.format({'encoder params': '{:,.0f}', 'table params': '{:,.0f}',
                                         'TRAINABLE': '{:,.0f}'}))
print('\nnew arms: %d -> %s  (%d runs)' % (len(NEW_ARMS), NEW_ARMS, len(NEW_ARMS) * len(SEEDS)))

---
## 5 · Data, probes and imports

In [ ]:
class RunData:
    '''Minimal TxData stand-in. The KG is intact in every arm of this notebook.'''

    def __init__(self, base, data_folder, split):
        self.df       = base['df']
        self.df_train = base['df_train']
        self.df_valid = base['df_valid']
        self.df_test  = base['df_test']
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


_SEED_CACHE = {}


def get_seed_data(seed):
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def encoder_reads_graph(model, G):
    '''Max |delta| in node embeddings when every edge_index is emptied. 0.0 => no message passing.'''
    model.eval()
    saved = {et: G[et].edge_index for et in G.edge_types}
    empty = torch.zeros((2, 0), dtype=torch.long, device=saved[next(iter(saved))].device)
    try:
        with torch.no_grad():
            h_full = model(G, G, return_h=True)
            for et in G.edge_types:
                G[et].edge_index = empty
            h_none = model(G, G, return_h=True)
        return float(max((h_full[nt] - h_none[nt]).abs().max().item() for nt in h_full))
    finally:
        for et, ei in saved.items():
            G[et].edge_index = ei


def embedding_learns(m):
    '''Max |change| in the node table after one finetune epoch. 0 => it receives no gradient.'''
    before = {k: v.detach().clone() for k, v in m.model.node_emb.items()}
    m.finetune(n_epoch=1, learning_rate=FINETUNE_CFG['learning_rate'],
               train_print_per_n=10_000, valid_per_n=10_000)
    return max(float((m.model.node_emb[k].detach() - v).abs().max()) for k, v in before.items())

In [ ]:
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import(arm_name, seed):
    if not IMPORT_REFS or arm_name not in IMPORTABLE:
        return False, 'not importable'
    a = ARMS_BY_NAME[arm_name]
    cfg = cfg_for(a['encoder'])
    expect = {'split': SPLIT, 'seed': seed,
              'model_config': {k: cfg[k] for k in MODEL_KEYS_CHECKED},
              'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG}
    return import_matching_run(
        a['source'] / ('%s_seed%d' % (arm_name, seed)), run_dir(arm_name, seed), expect,
        extra={'experiment': 'frozen_encoder', 'encoder': a['encoder'],
               'w_r_trained': not ENCODER_SPECS[a['encoder']]['freeze']})


if IMPORT_REFS:
    for _name in sorted(IMPORTABLE):
        n_ok = n_fail = 0
        for s in SEEDS:
            if is_done(_name, s):
                n_ok += 1
                continue
            ok, why = try_import(_name, s)
            if ok:
                n_ok += 1
            else:
                n_fail += 1
                if n_fail <= 2:
                    print('[import] %s_seed%d: will train (%s)' % (_name, s, why))
        print('[import] %-26s %2d present, %2d to train' % (_name, n_ok, n_fail))
else:
    print('IMPORT_REFS is False - every arm will be trained here.')

---
## 6 · The runner

| guard | catches |
|---|---|
| `type(m.model) is installed` | an encoder swap that did not take |
| `node_emb_params == n_nodes × n_inp` | a learnable table of the wrong size, or none at all |
| `emb_delta_after_1_epoch > 0` | a table that exists but never receives gradient |
| **`encoder_weight_delta == 0`** | **a "frozen" encoder that actually trained** |
| `trainable_params` matches the arm's budget, exact integers | a capacity-matched arm that is not |
| `graph_sensitivity` 0 for MLP arms, > 0 otherwise | an MLP that secretly reads edges |

In [ ]:
def run_one(arm_name, seed, verbose=True):
    arm  = ARMS_BY_NAME[arm_name]
    kind = arm['encoder']
    spec = ENCODER_SPECS[kind]
    cfg  = cfg_for(kind)
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    ok, why = try_import(arm_name, seed)
    if ok:
        print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
        return pd.read_csv(el_csv)

    print('\n' + '#' * 78)
    print('# %s   seed %d' % (arm_name, seed))
    print('# layer=%s  message passing=%s  learnable table=%s  W_r trained=%s  n_inp=%d'
          % (spec['layer'], spec['layer'] == 'rgcn', spec['emb'], not spec['freeze'], cfg['n_inp']))
    print('#' * 78)

    base = get_seed_data(seed)
    set_all_seeds(seed)
    d = RunData(base, DATA_FOLDER, SPLIT)
    rel_names = {et[1] for et in d.G.edge_types}
    assert all(r in rel_names for r in SIG_ETYPES_DEFAULT), 'KG is not intact'
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    n_nodes = sum(d.G[nt].num_nodes for nt in d.G.node_types)
    print('[graph] train edges %d | edge types %d | nodes %d'
          % (len(d.df_train), len(etypes_after), n_nodes))

    installed = install_encoder(kind)

    def build():
        set_all_seeds(seed)
        mm = TxGNN(data=d, weight_bias_track=False,
                   exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
        mm.model_initialize(**cfg)
        return mm

    m = build()
    assert type(m.model) is installed, 'encoder swap did not take: %s' % type(m.model).__name__
    layer_cls = type(m.model.layer1).__name__
    assert layer_cls == LAYER_CLASSES[spec['layer']].__name__, \
        'layer1 is %s, expected %s' % (layer_cls, LAYER_CLASSES[spec['layer']].__name__)

    enc_p, emb_p = encoder_params(m.model), node_emb_params(m.model)
    assert (emb_p > 0) == spec['emb'], \
        'learnable_emb=%s but node_emb_params=%d' % (spec['emb'], emb_p)
    if spec['emb']:
        assert emb_p == n_nodes * cfg['n_inp'], \
            'embedding table is %d, expected %d x %d' % (emb_p, n_nodes, cfg['n_inp'])

    # ---- the probe must run on a throwaway model, then rebuild ----
    emb_delta = None
    if PROBE_EMB_LEARNS and spec['emb']:
        if spec['freeze']:
            freeze_encoder(m.model, FREEZE_DECODER_W)
        set_all_seeds(seed)
        emb_delta = embedding_learns(m)
        print('[probe] embedding table moved by %.3g after one epoch' % emb_delta)
        assert emb_delta > 0, 'the embedding table received no gradient - it is not being trained'
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        m = build()

    # ---- freeze, for real this time ----
    n_frozen, snapshot = (0, None)
    if spec['freeze']:
        n_frozen, snapshot = freeze_encoder(m.model, FREEZE_DECODER_W)
        print('[freeze] %s encoder params frozen at init' % format(n_frozen, ','))

    train_p, tot_p = trainable_params(m.model), get_n_params(m.model)
    expect_train = (0 if spec['freeze'] else enc_p) + emb_p + 5000
    print('[model] encoder %s | table %s | TRAINABLE %s | total %s'
          % (format(enc_p, ','), format(emb_p, ','), format(train_p, ','), format(tot_p, ',')))
    assert train_p == expect_train, \
        'trainable params %d, expected %d' % (train_p, expect_train)

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- THE guard: a frozen encoder must be bit-identical to its init ----
    enc_delta = None
    if spec['freeze']:
        enc_delta = encoder_weight_delta(m.model, snapshot)
        print('[guard] max |change| in any encoder weight: %.3g  (must be 0)' % enc_delta)
        assert enc_delta == 0.0, 'the frozen encoder moved by %g - it was trained' % enc_delta

    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    gs = encoder_reads_graph(m.best_model, m.G)
    print('[probe] graph_sensitivity %.6g  (0 => no message passing)' % gs)
    if spec['layer'] == 'mlp':
        assert gs < 1e-6, '%s still reads the graph (delta %g)' % (arm_name, gs)
    else:
        assert gs > 1e-6, '%s should read the graph but does not' % arm_name

    meta = {
        'experiment': 'frozen_encoder',
        'run_key': rdir.name, 'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'note': arm['note'], 'encoder': kind, 'layer_class': layer_cls,
        'message_passing': spec['layer'] == 'rgcn',
        'learnable_emb': spec['emb'],
        'w_r_trained': not spec['freeze'],
        'decoder_w_frozen': bool(spec['freeze'] and FREEZE_DECODER_W),
        'n_frozen_params': int(n_frozen),
        'encoder_weight_delta': enc_delta,
        'emb_delta_after_1_epoch': emb_delta,
        'graph_sensitivity': gs,
        'n_nodes_total': int(n_nodes),
        'n_edge_types_after': len(etypes_after),
        'train_edges': int(len(d.df_train)),
        'model_config': cfg,
        'encoder_params': int(enc_p), 'node_emb_params': int(emb_p),
        'trainable_params': int(train_p), 'n_params': int(tot_p),
        'expect_trainable_params': int(expect_train),
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': round(pre_s, 1), 'finetune_seconds': round(fin_s, 1),
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    }
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d  indication %.4f  contraindication %.4f  micro %.4f  (%.1f min)'
          % (arm_name, seed, ind, con, micro_auprc, (pre_s + fin_s) / 60))

    install_encoder('rgcn')
    del m, d
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

---
## 7 · Run the grid

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending.' % len(todo))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 8 · Results

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'encoder_params': meta.get('encoder_params'),
                         'table_params': meta.get('node_emb_params'),
                         'trainable': meta.get('trainable_params', meta.get('n_params')),
                         'w_r_trained': meta.get('w_r_trained'),
                         'enc_delta': meta.get('encoder_weight_delta'),
                         'gs': meta.get('graph_sensitivity'),
                         'bce': meta.get('test_bce_loss'),
                         'n_inp': (meta.get('model_config') or {}).get('n_inp'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
wide = summary = None

if long.empty:
    print('No runs on disk yet - run section 7 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    _first = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
    info = long.groupby('arm').agg(encoder_params=('encoder_params', _first),
                                   table_params=('table_params', _first),
                                   trainable=('trainable', _first),
                                   w_r_trained=('w_r_trained', _first),
                                   enc_delta=('enc_delta', _first),
                                   gs=('gs', _first), n_inp=('n_inp', _first))

    rows = []
    ref_w = wide.xs(REF_ARM, level='arm') if REF_ARM in wide.index.get_level_values('arm') else None
    for a in wide.index.get_level_values('arm').unique():
        w = wide.xs(a, level='arm')
        r = {'arm': a, 'n_seeds': len(w)}
        for k in ROW:
            r[k] = w[k].mean()
            r['%s_ci' % k] = _ci95(w[k])
        if ref_w is not None and a != REF_ARM:
            common = w.index.intersection(ref_w.index)
            for k in ROW:
                dl = (w.loc[common, k] - ref_w.loc[common, k]).dropna()
                r['%s_delta' % k] = dl.mean()
                r['%s_delta_ci' % k] = _ci95(dl)
                r['%s_hurt' % k] = int((dl < 0).sum())
        rows.append(r)
    summary = (pd.DataFrame(rows).merge(info.reset_index(), on='arm', how='left')
               .sort_values('indication', ascending=False))

    print('Absolute AUPRC (chance = 0.500 on this protocol)')
    display(summary[['arm', 'n_seeds', 'w_r_trained', 'n_inp', 'trainable', 'enc_delta', 'gs',
                     'indication', 'indication_ci', 'contraindication', 'contraindication_ci',
                     'micro', 'micro_ci']]
            .style.format({'indication': '{:.4f}', 'indication_ci': '±{:.4f}',
                           'contraindication': '{:.4f}', 'contraindication_ci': '±{:.4f}',
                           'micro': '{:.4f}', 'micro_ci': '±{:.4f}',
                           'trainable': '{:,.0f}', 'n_inp': '{:.0f}',
                           'enc_delta': '{:.3g}', 'gs': '{:.3g}'}, na_rep='-'))

### 8.1 The contrasts that matter

`frozen − trained` is the value of letting the encoder learn, with the node table learnable on both
sides. A null here means the per-relation weight matrices contribute nothing once node embeddings are
free parameters.

In [ ]:
def paired(a, b, label):
    '''Per-seed (a - b) with a t-CI and a sign count. None if either arm is absent.'''
    if wide is None:
        return None
    ix = wide.index.get_level_values('arm').unique()
    if a not in ix or b not in ix:
        return None
    wa, wb = wide.xs(a, level='arm'), wide.xs(b, level='arm')
    common = wa.index.intersection(wb.index)
    if not len(common):
        return None
    out = {'contrast': label, 'n': len(common)}
    for k in ROW:
        d = (wa.loc[common, k] - wb.loc[common, k]).dropna()
        out[k] = d.mean()
        out['%s_ci' % k] = _ci95(d)
        out['%s_favours_a' % k] = int((d > 0).sum())
    return out


CONTRASTS = [
    ('emb_rgcn_frozenW',         'emb_rgcn', 'R-GCN: frozen W_r - trained W_r'),
    ('emb_rgcn_frozenW_matched', 'emb_rgcn', 'R-GCN: frozen W_r, matched budget - trained'),
    ('emb_mlp_frozenW',          'emb_mlp',  'MLP: frozen weight - trained weight'),
    ('emb_rgcn_frozenW',         'emb_mlp_frozenW',
     'message passing through a RANDOM projection'),
    ('emb_rgcn_frozenW_matched', 'emb_rgcn_frozenW', 'the matched budget alone'),
    ('emb_rgcn_frozenW',         'rgcn',     'frozen W_r + table - published model'),
]

rows = [r for r in (paired(a, b, lbl) for a, b, lbl in CONTRASTS) if r]
if rows:
    display(pd.DataFrame(rows)[['contrast', 'n', 'indication', 'indication_ci',
                                'indication_favours_a', 'contraindication',
                                'contraindication_ci', 'micro', 'micro_ci']]
            .style.format({'indication': '{:+.4f}', 'indication_ci': '±{:.4f}',
                           'contraindication': '{:+.4f}', 'contraindication_ci': '±{:.4f}',
                           'micro': '{:+.4f}', 'micro_ci': '±{:.4f}'}, na_rep='-'))
    print('\n"favours_a" counts the seeds on which the first arm scored higher, out of n.')
else:
    print('Not enough arms finished for these contrasts yet.')

### 8.2 Guard audit — did the frozen arms really stay frozen?

In [ ]:
if long.empty:
    print('No runs yet.')
else:
    g = (long.groupby('arm')
         .agg(n=('seed', 'nunique'), w_r_trained=('w_r_trained', 'first'),
              enc_delta_max=('enc_delta', 'max'), gs=('gs', 'first'),
              trainable=('trainable', 'first'))
         .reset_index())
    display(g.style.format({'enc_delta_max': '{:.3g}', 'gs': '{:.3g}',
                            'trainable': '{:,.0f}'}, na_rep='-'))
    frozen = g[g.w_r_trained == False]
    if len(frozen):
        bad = frozen[frozen.enc_delta_max.fillna(1) != 0]
        if len(bad):
            print('FAIL: these frozen arms moved: %s' % ', '.join(bad.arm))
        else:
            print('OK: every frozen arm ended with encoder_weight_delta exactly 0 on every seed.')
    if summary is not None:
        out_csv = ABL_ROOT / 'frozen_encoder_summary.csv'
        summary.to_csv(out_csv, index=False)
        wide.reset_index().to_csv(ABL_ROOT / 'frozen_encoder_per_seed.csv', index=False)
        print('wrote', out_csv.relative_to(REPO_ROOT))

---
## 9 · What this grid can and cannot support

**Can.**

- Whether the encoder's learned weights matter at all once node embeddings are free parameters. Both
  sides of `emb_rgcn_frozenW − emb_rgcn` have a learnable table; only the encoder differs.
- Message passing through an *untrained* projection (`emb_rgcn_frozenW − emb_mlp_frozenW`), at an
  identical trainable budget by construction. If this is positive while the trained contrast is null,
  then aggregation over neighbours — not the learned mixing weights — is what message passing buys.
- The same question for a model with no message passing: `emb_mlp_frozenW − emb_mlp` asks whether the
  single shared 20,200-parameter `Linear` was ever worth training.

**Cannot.**

- "Frozen at random init" is not "absent". A fixed random projection still mixes neighbours and still
  spends 1,010,000 numbers doing it; a null result says those numbers need not be *learned*, not that
  they could be removed. `light_gnn`-style arms are the ones that remove capacity.
- Matched *trainable* parameters is not matched *capacity*. The frozen arms carry 1,010,000 fixed
  numbers that contribute to the function but not to the count, so `emb_rgcn_frozenW` is arguably
  richer than its trainable count suggests, in a way `n_inp = 108` does not correct.
- The decoder stays trainable (`FREEZE_DECODER_W = False`), so this is not a claim about an entirely
  untrained model — the DistMult relation vectors still learn.
- Widening `n_inp` to 108 in the matched arm also widens the frozen input projection, so that arm
  differs from `emb_rgcn_frozenW` in two ways at once. Read it as a capacity control on the headline
  contrast, not as a clean third point.
- Edge-level AUPRC only, chance 0.500. Nothing here speaks to the ranking protocol.